In [10]:
import os
import sys

sys.path.insert(0, os.path.abspath("E:/Anaconda3/envs/Judge-Assistant"))

from langchain_huggingface import HuggingFaceEmbeddings
from RAG.civil_law_rag.config import EMBEDDING_MODEL

print(f"Loading model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large-instruct")
# embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Test 1: basic embedding
vec = embeddings.embed_query("ما هي شروط صحة العقد؟")
print(f"Vector length: {len(vec)}")
print(f"First 5 values: {vec[:5]}")
print(f"All zeros? {all(v == 0 for v in vec)}")

# Test 2: semantic similarity check
from numpy import dot
from numpy.linalg import norm

def cosine(a, b):
    return dot(a, b) / (norm(a) * norm(b))

v1 = embeddings.embed_query("شروط صحة العقد")
v2 = embeddings.embed_query("أركان العقد الصحيح")   # semantically close
v3 = embeddings.embed_query("ما هو أفضل مطعم في القاهرة")  # off-topic

print(f"\nSimilarity (contract conditions vs contract elements): {cosine(v1, v2):.4f}")  # should be HIGH > 0.8
print(f"Similarity (contract conditions vs restaurant):        {cosine(v1, v3):.4f}")  # should be LOW < 0.5

Loading model: intfloat/multilingual-e5-large-instruct


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector length: 1024
First 5 values: [0.00446319580078125, 0.014984130859375, -0.01380157470703125, -0.030792236328125, 0.031494140625]
All zeros? False

Similarity (contract conditions vs contract elements): 0.9582
Similarity (contract conditions vs restaurant):        0.7905


In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchAny

client = QdrantClient(host="localhost", port=6333)

# Fetch articles 89, 90, 147
results, _ = client.scroll(
    collection_name="civil_law_docs",
    scroll_filter=Filter(must=[
        FieldCondition(
            key="metadata.index",
            match=MatchAny(any=[89, 90, 147])
        )
    ]),
    with_payload=True,
    limit=3
)

for r in results:
    idx = r.payload.get("metadata", {}).get("index")
    text = r.payload.get("page_content", "")
    print(f"\n=== Article {idx} ===")
    print(text[:300])


=== Article 89 ===
المادة 89

89
يتم العقد بمجرد أن يتبادل طرفان التعبير عن إرادتين متطابقتين، مع مراعاة ما يقرره القانون من أوضاع معينة لانعقاد العقد.

=== Article 147 ===
المادة 147

147
العقد شريعة المتعاقدين، فلا يجوز نقضه أو تعديله إلا باتفاق الطرفين أو لأسباب يقررها القانون.

إذا طرأت حوادث استثنائية لم يكن في الوسع توقعها، وترتب على حدوثها أن تنفيذ الالتزام صار مرهقًا للمدين، جاز للقاضي بعد الموازنة بين مصالح الطرفين أن يرد الالتزام إلى الحد المعقول. ويكون أي ات

=== Article 90 ===
المادة 90

90
التعبير عن الإرادة يكون باللفظ والكتابة وبالإشارة المتداولة عرفًا، كما يكون باتخاذ موقف لا تدع ظروف الحال شكا في دلالته على حقيقة المقصود.

يجوز أن يكون التعبير عن الإرادة ضمنيًا إذا لم ينص القانون أو يتفق الطرفان على ذلك.


In [5]:
from numpy import dot
from numpy.linalg import norm

def cosine(a, b):
    return dot(a, b) / (norm(a) * norm(b))

# paste the actual article texts from the output above
art_89_text  = "..."  
art_147_text = "..."  

q_a001 = embeddings.embed_query("ما هي شروط صحة العقد في القانون المدني المصري؟")
q_a008 = embeddings.embed_query("ما هي شروط تطبيق نظرية الظروف الطارئة؟")

v_89  = embeddings.embed_query(art_89_text[:500])
v_147 = embeddings.embed_query(art_147_text[:500])

print(f"A001 query ↔ article 89:  {cosine(q_a001, v_89):.4f}")   # should be high
print(f"A008 query ↔ article 147: {cosine(q_a008, v_147):.4f}")  # should be high

A001 query ↔ article 89:  0.2846
A008 query ↔ article 147: 0.2911


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from numpy import dot
from numpy.linalg import norm

def cosine(a, b):
    return dot(a, b) / (norm(a) * norm(b))

art_89  = "المادة 89\nيتم العقد بمجرد أن يتبادل طرفان التعبير عن إرادتين متطابقتين"
art_147 = "المادة 147\nإذا طرأت حوادث استثنائية لم يكن في الوسع توقعها، وترتب على حدوثها أن تنفيذ الالتزام صار مرهقًا للمدين"

q_a001 = "ما هي شروط صحة العقد في القانون المدني المصري؟"
q_a008 = "ما هي شروط تطبيق نظرية الظروف الطارئة؟"

for model_name in [
    "intfloat/multilingual-e5-large-instruct",
    "Omartificial/Arabic-Mpnet-Base-all-nli-triplet",
    "CAMeL-Lab/bert-base-arabic-camelbert-msa",
]:
    print(f"\n=== {model_name} ===")
    try:
        emb = HuggingFaceEmbeddings(model_name=model_name)
        v89  = emb.embed_query(art_89)
        v147 = emb.embed_query(art_147)
        va001 = emb.embed_query(q_a001)
        va008 = emb.embed_query(q_a008)
        print(f"A001 ↔ art89:  {cosine(va001, v89):.4f}")
        print(f"A008 ↔ art147: {cosine(va008, v147):.4f}")
    except Exception as e:
        print(f"Failed: {e}")


=== intfloat/multilingual-e5-large-instruct ===


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

A001 ↔ art89:  0.6535
A008 ↔ art147: 0.6489


In [1]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("intfloat/multilingual-e5-large-instruct")
model = model.cuda()

total = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3
print(f"Model VRAM: {total:.2f} GB")
print(f"Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu118


In [6]:
print(torch.__version__)       # should show 2.x.x+cu118

2.11.0+cpu


In [7]:
# First check which python the notebook is actually using
import sys
print(sys.executable)

e:\Anaconda3\envs\ai-judge\python.exe


In [8]:
!E:\Anaconda3\envs\ai-judge\python.exe -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu118


In [9]:
!E:\Anaconda3\envs\ai-judge\python.exe -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 1.9 MB/s eta 0:25:16
   ---------------------------------------- 0.0/2.8 GB 1.9 MB/s eta 0:24:14
   ---------------------------------------- 0.0/2.8 GB 2.0 MB/s eta 0:23:46
   ---------------------------------------- 0.0/2.8 GB 1.9 MB/s eta 0:24:14
   ---------------------------------------- 0.0/2.8 GB 1.9 MB/s eta 0:24:07
   ---------------------------------------- 

ERROR: Exception:
Traceback (most recent call last):
  File "E:\Anaconda3\envs\ai-judge\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "E:\Anaconda3\envs\ai-judge\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda3\envs\ai-judge\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda3\envs\ai-judge\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 102, in read
    self.__buf.write(data)
  File "E:\Anaconda3\envs\ai-judge\Lib\tempfile.py", line 500, in func_wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 28] No space left on device

During handling of the above exception, another exception occurred:

Traceback (most recent 

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from collections import Counter

client = QdrantClient(host="localhost", port=6333)
results, _ = client.scroll(
    collection_name="civil_law_docs",
    scroll_filter=Filter(must=[
        FieldCondition(key="metadata.source", match=MatchValue(value="civil_law")),
        FieldCondition(key="metadata.type", match=MatchValue(value="article")),
    ]),
    with_payload=True,
    with_vectors=False,
    limit=2000,
)

section_counts = Counter(
    r.payload.get("metadata", {}).get("section", "NO_SECTION")
    for r in results
)
for section, count in section_counts.most_common(10):
    print(f"{count:3d} articles — {section}")

176 articles — None
 42 articles — ١ – أركان العقد
 29 articles — آثار الإيجار
 28 articles — التزامات البائع
 26 articles — حق التقدم وحق التتبع
 23 articles — الشخص الطبيعي
 21 articles — ١ – التضامن
 19 articles — ٢ – القيود التي ترد على حق الملكية
 19 articles — ٣ – تنازع القوانين من حيث المكان
 19 articles — أحكام عامة
